# Day 2 | Lab 2.4: LangChain Output Parsers Deep-Dive

**Duration:** ~1.5 hours

**Scenario.** Domain-mixed examples (CRM extraction, e-commerce product tagging, customer-support triage, loan applications) — preserved from the GM source. The lab focuses on the parsing layer; scenarios are vehicles for the technical points.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Use `StrOutputParser` for clean string extraction and streaming.
2. Pick the right parser per task: `CommaSeparatedListOutputParser`, `DatetimeOutputParser`, `StructuredOutputParser`, `JsonOutputParser`, `PydanticOutputParser`, `EnumOutputParser`.
3. Use `OutputFixingParser` to recover from malformed model output, and recognize when it's a code smell.
4. Choose between `Pydantic + .with_structured_output()` (modern preferred path) and `PydanticOutputParser` (works on any model).
5. Enforce **XML-tagged output** via system prompt alone — Claude's preferred pattern, no parser library needed.
6. Compose a real ETL pipeline: `LLM → Parser → business rules → LLM` with typed objects throughout.

*Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---


## 1. Install Dependencies

In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q 'langchain>=1.0' 'langchain-core>=1.0' 'langchain-openai>=1.0' 'langchain-anthropic>=0.3' 'langchain-groq>=0.2' 'langchain-community>=0.3' pydantic


## 2. API Key Configuration

In [ ]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

for key in ['OPENAI_API_KEY', 'GROQ_API_KEY', 'ANTHROPIC_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


In [ ]:
import textwrap

def wrap_print(text, width=100):
    """Prints text wrapped to a specified width."""
    print(textwrap.fill(text, width=width))
    print()

# Example usage with a long string
long_text = "This is a very long string that would normally require horizontal scrolling to read completely if it were just printed out directly to the console without any formatting. But with this helper function, it will be nicely wrapped!"

wrap_print(long_text)

This is a very long string that would normally require horizontal scrolling to read completely if it
were just printed out directly to the console without any formatting. But with this helper function,
it will be nicely wrapped!



## 3. Initialize LLMs

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

# Primary: OpenAI gpt-4.1-mini (supports temperature)
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.2)

# Secondary: Groq (open-source Llama 3.3)
llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)

print("LLMs initialized ✅")

LLMs initialized ✅


---
## 4. StrOutputParser — Plain Text Extraction

### Business Scenario: Customer Email Reply Generator
A retail bank's support team uses an LLM to draft email replies. The output just needs to be clean text — no special structure. `StrOutputParser` extracts the text content from the LLM's `AIMessage` object.

> This is the most commonly used parser. It converts `AIMessage` → `str`, which is essential for chaining (most downstream components expect strings, not message objects).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Without StrOutputParser — returns AIMessage object
prompt = ChatPromptTemplate.from_template(
    "Draft a short, polite reply to this customer complaint about a delayed credit card delivery: {complaint}"
)
raw_chain = prompt | llm
raw_result = raw_chain.invoke({"complaint": "I applied 3 weeks ago and still haven't received my card."})
print(f"Type WITHOUT parser: {type(raw_result)}")
print(f"Content: {raw_result.content[:100]}...\n")

Type WITHOUT parser: <class 'langchain_core.messages.ai.AIMessage'>
Content: Dear [Customer Name],

Thank you for reaching out. We apologize for the delay in delivering your cre...



In [ ]:
raw_result

AIMessage(content='Dear [Customer Name],\n\nThank you for reaching out. We apologize for the delay in delivering your credit card. We are looking into this matter and will update you shortly. Please feel free to contact us if you have any further questions.\n\nBest regards,  \n[Your Name]  \n[Your Company]', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 37, 'total_tokens': 97, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_a391f2cee0', 'id': 'chatcmpl-DDAvNjSnWd7TdNsA2KAtaIA7vHaJo', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c9560-12cd-77d3-aaf7-0a3c67264c4b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_token

In [ ]:
# With StrOutputParser — returns clean string
parsed_chain = prompt | llm | StrOutputParser()
parsed_result = parsed_chain.invoke({"complaint": "I applied 3 weeks ago and still haven't received my card."})
print(f"Type WITH parser: {type(parsed_result)}")
wrap_print(f"Content: {parsed_result}")

Type WITH parser: <class 'langchain_core.messages.base.TextAccessor'>
Content: Dear [Customer Name],  Thank you for reaching out. We apologize for the delay in delivering
your credit card. We are looking into this matter and will update you shortly. Please feel free to
contact us if you have any further questions.  Best regards,   [Your Name]   [Your Company]



In [ ]:
# --- StrOutputParser + Streaming (the primary use case for this parser) ---
# Streaming requires StrOutputParser to yield text chunks instead of AIMessage chunks

stream_chain = prompt | llm | StrOutputParser()

print("Streaming reply token by token:")
for chunk in stream_chain.stream({"complaint": "I was charged twice for my last EMI payment."}):
    print(chunk, end="", flush=True)
print("\n\n✅ Streaming works because StrOutputParser converts each AIMessageChunk → str chunk")

Streaming reply token by token:
Dear [Customer Name],

Thank you for reaching out. We apologize for the inconvenience caused by the double charge on your last EMI payment. We are looking into this issue and will resolve it as quickly as possible. 

Please allow us some time to investigate, and we will update you shortly.

Thank you for your patience and understanding.

Best regards,  
[Your Name]  
[Your Company]

✅ Streaming works because StrOutputParser converts each AIMessageChunk → str chunk


---
## 5. CommaSeparatedListOutputParser — Lists from LLMs

### Business Scenario: Product Tag Generator for E-Commerce
An e-commerce platform auto-generates searchable tags for new product listings. The system needs a **Python list** — not a paragraph of text — to store tags in the product database.

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate

# Initialize the parser
list_parser = CommaSeparatedListOutputParser()

# See what instructions the parser generates for the LLM
print("📋 Format instructions sent to LLM:")
print(list_parser.get_format_instructions())
print()

📋 Format instructions sent to LLM:
Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`



In [ ]:
# Build the chain — format_instructions are injected as a partial variable
tag_prompt = PromptTemplate(
    template="Generate 5-8 relevant search tags for this product listing.\n\n"
             "Product: {product_description}\n\n{format_instructions}",
    input_variables=["product_description"],
    partial_variables={"format_instructions": list_parser.get_format_instructions()}
)

tag_chain = tag_prompt | llm | list_parser

# Test with real product descriptions
products = [
    "Nike Air Max 270 Men's Running Shoes — Black/White, breathable mesh, cushioned sole, size 10",
    "Organic Darjeeling First Flush Tea — 100g loose leaf, hand-picked, USDA certified, floral aroma",
    "Samsung 55-inch QLED 4K Smart TV — HDR10+, Tizen OS, gaming mode, wall-mountable"
]

for desc in products:
    tags = tag_chain.invoke({"product_description": desc})
    print(f"🏷️ {desc}")
    wrap_print(f"   Type: {type(tags)} | Tags: {tags}")
    print()

🏷️ Nike Air Max 270 Men's Running Shoes — Black/White, breathable mesh, cushioned sole, size 10
   Type: <class 'list'> | Tags: ['Nike Air Max 270', "men's running shoes", 'black white sneakers',
'breathable mesh shoes', 'cushioned sole sneakers', 'size 10 running shoes', 'Nike athletic
footwear', "men's sports shoes"]


🏷️ Organic Darjeeling First Flush Tea — 100g loose leaf, hand-picked, USDA certified, floral aroma
   Type: <class 'list'> | Tags: ['organic darjeeling tea', 'first flush tea', 'loose leaf tea',
'hand-picked tea', 'USDA certified tea', 'floral aroma tea', 'premium darjeeling', '100g tea
leaves']


🏷️ Samsung 55-inch QLED 4K Smart TV — HDR10+, Tizen OS, gaming mode, wall-mountable
   Type: <class 'list'> | Tags: ['Samsung QLED TV', '55-inch 4K TV', 'Smart TV', 'HDR10+', 'Tizen
OS', 'gaming mode TV', 'wall-mountable TV', '4K UHD TV']




In [ ]:
tags   # exact output for the last product description

['Samsung QLED TV',
 '55-inch 4K TV',
 'Smart TV',
 'HDR10+',
 'Tizen OS',
 'gaming mode TV',
 'wall-mountable TV',
 '4K UHD TV']

> **When to use:** Whenever you need a simple flat list — product tags, categories, feature lists, skill lists, keywords. For structured data with multiple fields, use JSON or Pydantic parsers instead.

---
## 6. DatetimeOutputParser — Date Extraction

### Business Scenario: Insurance Policy Date Extractor
An insurance company processes customer emails to extract policy-related dates (start date, renewal, incident date). These need to be Python `datetime` objects for database storage and compliance calculations.

In [ ]:
# pip install langchain langchain-core langchain-community

In [ ]:
from langchain_core.output_parsers import DatetimeOutputParser

datetime_parser = DatetimeOutputParser()

# See what format it expects
print("📋 Format instructions:")
print(datetime_parser.get_format_instructions())
print()

ModuleNotFoundError: No module named 'langchain_core.output_parsers.datetime'

In [ ]:
date_prompt = PromptTemplate(
    template="Extract the exact date from this insurance-related query. "
             "If multiple dates, extract the most important one.\n\n"
             "Query: {query}\n\n{format_instructions}",
    input_variables=["query"],
    partial_variables={"format_instructions": datetime_parser.get_format_instructions()}
)

date_chain = date_prompt | llm | datetime_parser

# Test with various date formats people use naturally
queries = [
    "My car accident happened on March 15th, 2024 around 3pm.",
    "Policy renewal is due next week — I think it's January 20, 2025.",
    "The house fire occurred on the night of December 31, 2023."
]

for q in queries:
    try:
        parsed_date = date_chain.invoke({"query": q})
        print(f"📅 Query: {q[:50]}...")
        print(f"   Parsed: {parsed_date} | Type: {type(parsed_date)}")
        print(f"   Year: {parsed_date.year}, Month: {parsed_date.month}, Day: {parsed_date.day}")
    except Exception as e:
        print(f"❌ Failed to parse: {e}")
    print()

> **When to use:** Whenever you need to extract dates from free-text and convert them to Python `datetime` objects — scheduling systems, compliance date tracking, report date extraction.

In [ ]:
from datetime import datetime

type(datetime.now())

datetime.datetime

In [ ]:
datetime.now()

datetime.datetime(2026, 2, 25, 15, 38, 13, 593514)

In [ ]:
import datetime
import pytz

# Get the current UTC time
utc_now = datetime.datetime.utcnow().replace(tzinfo=pytz.utc)
print(f"UTC current time: {utc_now}")

# Define the target timezone
kolkata_tz = pytz.timezone('Asia/Kolkata')

# Localize the UTC time to Asia/Kolkata
kolkata_now = utc_now.astimezone(kolkata_tz)

print(f"Asia/Kolkata current time: {kolkata_now}")
print(f"Timezone name: {kolkata_now.tzname()}")
print(f"Timezone offset: {kolkata_now.utcoffset()}")

UTC current time: 2026-02-25 15:39:49.546505+00:00
Asia/Kolkata current time: 2026-02-25 21:09:49.546505+05:30
Timezone name: IST
Timezone offset: 5:30:00


/tmp/ipython-input-3158324521.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  utc_now = datetime.datetime.utcnow().replace(tzinfo=pytz.utc)


In [ ]:
from datetime import datetime
datetime.now().strftime("%y-%b-%d %H:%M:%S")

'26-Feb-25 15:31:46'

In [ ]:
from datetime import datetime, timedelta

current_date = datetime.now()
date_30_days_before = current_date - timedelta(days=30)

print(f"Current date: {current_date.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Date 30 days before: {date_30_days_before.strftime('%Y-%m-%d %H:%M:%S')}")

Current date: 2026-02-25 15:33:04
Date 30 days before: 2026-01-26 15:33:04


---
## 7. StructuredOutputParser — Quick Structured Dicts

### Business Scenario: Sales Call Summary Extraction
A sales team records call notes as free text. The CRM system needs structured data: customer name, interest level, next action, and follow-up date. `StructuredOutputParser` defines the schema using `ResponseSchema` objects — simpler than Pydantic when you just need a flat dict.

In [ ]:
from langchain_community.output_parsers.structured import StructuredOutputParser, ResponseSchema

# Define the fields we want extracted
response_schemas = [
    ResponseSchema(name="customer_name", description="Full name of the customer/prospect"),
    ResponseSchema(name="company", description="Company name of the prospect"),
    ResponseSchema(name="interest_level", description="hot, warm, or cold"),
    ResponseSchema(name="product_interest", description="Which product/service they showed interest in"),
    ResponseSchema(name="next_action", description="Recommended next step for the sales rep"),
    ResponseSchema(name="follow_up_date", description="Suggested follow-up date in YYYY-MM-DD format"),
]

struct_parser = StructuredOutputParser.from_response_schemas(response_schemas)

# See the auto-generated format instructions
print("📋 Format instructions (sent to the LLM):")
print(struct_parser.get_format_instructions())

ImportError: cannot import name 'StructuredOutputParser' from 'langchain_core.output_parsers' (/usr/local/lib/python3.12/dist-packages/langchain_core/output_parsers/__init__.py)

In [ ]:
crm_prompt = PromptTemplate(
    template="Extract CRM data from these sales call notes.\n\n"
             "Call Notes: {call_notes}\n\n{format_instructions}",
    input_variables=["call_notes"],
    partial_variables={"format_instructions": struct_parser.get_format_instructions()}
)

crm_chain = crm_prompt | llm | struct_parser

# Test with real-ish sales call notes
call_notes = [
    """Had a great call with Rajesh Khanna, CTO of MedTech Solutions. Very interested in our
    Enterprise RAG platform for their medical records search. Wants a demo next week.
    Budget approved for Q2. Should schedule a technical deep-dive by Feb 20th.""",

    """Quick intro call with Priya Singh from QuickRetail. She's evaluating chatbot solutions
    for their customer support. Still early stage — comparing 3 vendors. Asked us to send
    pricing docs. Low urgency, follow up in 2 weeks around March 1st.""",
]

for notes in call_notes:
    result = crm_chain.invoke({"call_notes": notes})
    print(f"\n📞 Call Notes: {notes[:60]}...")
    print(f"   Type: {type(result)}")
    for key, val in result.items():
        print(f"   {key:20s}: {val}")
    print("-" * 60)

> **When to use:** When you need a flat dictionary with named fields but don't want to define a full Pydantic model. Great for quick prototyping and CRM/form-filling use cases. For nested structures or strict type validation, use `PydanticOutputParser` instead.

---
## 8. JsonOutputParser — Flexible JSON Output

### Business Scenario: Healthcare API — Patient Triage Response
A hospital's triage API receives symptom descriptions and returns structured JSON responses for the frontend app. `JsonOutputParser` can work **with or without a Pydantic schema** — giving you flexibility to enforce a schema when needed.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List

# --- 8a. JsonOutputParser WITHOUT schema (free-form JSON) ---
# Useful for exploration — the LLM decides the structure

json_parser_free = JsonOutputParser()   # No schema

triage_prompt_free = PromptTemplate(
    template="You are a medical triage assistant. Analyze the symptoms and return a JSON response "
             "with severity, likely conditions, and recommended action.\n\n"
             "Symptoms: {symptoms}\n\n{format_instructions}",
    input_variables=["symptoms"],
    partial_variables={"format_instructions": json_parser_free.get_format_instructions()}
)

free_json_chain = triage_prompt_free | llm | json_parser_free

result = free_json_chain.invoke({"symptoms": "Sharp chest pain, shortness of breath, sweating for the last 30 minutes"})
print("📋 Free-form JSON (no schema):")
print(f"   Type: {type(result)}")
import json
print(json.dumps(result, indent=2))

📋 Free-form JSON (no schema):
   Type: <class 'dict'>
{
  "severity": "high",
  "likely_conditions": [
    "acute myocardial infarction (heart attack)",
    "pulmonary embolism",
    "unstable angina"
  ],
  "recommended_action": "Call emergency services immediately or go to the nearest emergency room for urgent evaluation and treatment."
}


In [ ]:
# --- 8b. JsonOutputParser WITH Pydantic schema (enforced structure) ---
# The schema is sent to the LLM as format instructions, ensuring consistent output

class TriageResponse(BaseModel):
    severity: str = Field(description="One of: CRITICAL, HIGH, MODERATE, LOW")
    likely_conditions: List[str] = Field(description="Top 2-3 possible conditions")
    recommended_action: str = Field(description="Immediate recommended action")
    urgency_minutes: int = Field(description="How urgently patient should be seen (in minutes)")

json_parser_schema = JsonOutputParser(pydantic_object=TriageResponse)

# See the schema-aware format instructions
print("📋 Schema-aware format instructions:")
print()
print(json_parser_schema.get_format_instructions())
# print("...")

📋 Schema-aware format instructions:

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any ba

In [ ]:
triage_prompt_schema = PromptTemplate(
    template="You are a medical triage assistant. Analyze the symptoms and respond with structured JSON.\n\n"
             "Symptoms: {symptoms}\n\n{format_instructions}",
    input_variables=["symptoms"],
    partial_variables={"format_instructions": json_parser_schema.get_format_instructions()}
)

schema_json_chain = triage_prompt_schema | llm | json_parser_schema

# Test with different symptom severities
test_symptoms = [
    "Sharp chest pain, shortness of breath, sweating for the last 30 minutes",
    "Mild headache and runny nose for 2 days, no fever",
    "Twisted ankle while playing cricket, moderate swelling, can still walk with pain"
]

for symptoms in test_symptoms:
    result = schema_json_chain.invoke({"symptoms": symptoms})
    print(f"\n🏥 Symptoms: {symptoms}")
    print(result)
    print(f"   Type: {type(result)}")
    print(f"   Severity: {result['severity']} | Urgency: {result['urgency_minutes']}min")
    print(f"   Conditions: {result['likely_conditions']}")
    print(f"   Action: {result['recommended_action']}")
    print("-" * 60)


🏥 Symptoms: Sharp chest pain, shortness of breath, sweating for the last 30 minutes
{'severity': 'CRITICAL', 'likely_conditions': ['Myocardial infarction (heart attack)', 'Pulmonary embolism', 'Acute coronary syndrome'], 'recommended_action': 'Call emergency services immediately or go to the nearest emergency department.', 'urgency_minutes': 0}
   Type: <class 'dict'>
   Severity: CRITICAL | Urgency: 0min
   Conditions: ['Myocardial infarction (heart attack)', 'Pulmonary embolism', 'Acute coronary syndrome']
   Action: Call emergency services immediately or go to the nearest emergency department.
------------------------------------------------------------

🏥 Symptoms: Mild headache and runny nose for 2 days, no fever
{'severity': 'LOW', 'likely_conditions': ['Common cold', 'Allergic rhinitis', 'Mild viral infection'], 'recommended_action': 'Rest, stay hydrated, and monitor symptoms. Use over-the-counter remedies for symptom relief if needed.', 'urgency_minutes': 1440}
   Type: <class

> **Key Difference:** `JsonOutputParser` always returns a **Python dict**. Use it without a schema for exploration, with a Pydantic schema for consistency. For full Pydantic validation (type checking, default values, nested models), use `PydanticOutputParser` instead.

---
## 9. PydanticOutputParser — Type-Safe Validated Output

### Business Scenario: Loan Application Pre-Screening
A bank's loan system extracts applicant data from free-text descriptions, validates it (age must be int, income must be float, etc.), and feeds it into the credit scoring engine. `PydanticOutputParser` gives you **full type validation, default values, and nested models** — critical for production data pipelines.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional

# --- Define a nested Pydantic model with validation ---
class EmploymentDetails(BaseModel):
    employer: str = Field(description="Current employer name")
    designation: str = Field(description="Job title/designation")
    years_in_current_job: float = Field(description="Years at current employer")

class LoanApplicant(BaseModel):
    full_name: str = Field(description="Applicant's full name")
    age: int = Field(description="Applicant's age in years")
    monthly_income: float = Field(description="Monthly income in INR")
    existing_emis: float = Field(default=0.0, description="Total existing monthly EMIs in INR")
    loan_amount_requested: float = Field(description="Loan amount requested in INR")
    loan_purpose: str = Field(description="Purpose: home, car, personal, education, or business")
    employment: EmploymentDetails = Field(description="Current employment details")
    cibil_score: Optional[int] = Field(default=None, description="CIBIL score if mentioned, else null")

# Create the parser
pydantic_parser = PydanticOutputParser(pydantic_object=LoanApplicant)

print("📋 Pydantic format instructions (truncated):")
print(pydantic_parser.get_format_instructions()[:400])
print("...")

📋 Pydantic format instructions (truncated):
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is n
...


In [ ]:
loan_prompt = PromptTemplate(
    template="Extract loan application details from this text. "
             "If a field is not mentioned, use a reasonable default or null.\n\n"
             "Application Text: {application_text}\n\n{format_instructions}",
    input_variables=["application_text"],
    partial_variables={"format_instructions": pydantic_parser.get_format_instructions()}
)

loan_chain = loan_prompt | llm | pydantic_parser

# Test with realistic applicant descriptions
applications = [
    """I'm Amit Verma, 34 years old, working as a Senior Software Engineer at Infosys for the
    last 4 years. My monthly salary is ₹1,85,000. I'm paying ₹15,000 EMI for my car loan.
    I want a home loan of ₹75 lakhs. My CIBIL score is 780.""",

    """Sunita Reddy, age 28, Product Manager at Flipkart for 2.5 years. Earning ₹1,20,000 per month.
    No existing loans. Looking for a personal loan of ₹5,00,000 for my sister's wedding."""
]

for app_text in applications:
    result = loan_chain.invoke({"application_text": app_text})

    print(f"\n👤 Parsed Applicant: {result.full_name}")
    print(f"   Type: {type(result)} (Pydantic model — fully validated!)")
    print(f"   Age: {result.age} | Income: ₹{result.monthly_income:,.0f}")
    print(f"   Employer: {result.employment.employer} | Role: {result.employment.designation}")
    print(f"   Loan: ₹{result.loan_amount_requested:,.0f} ({result.loan_purpose})")
    print(f"   Existing EMIs: ₹{result.existing_emis:,.0f} | CIBIL: {result.cibil_score}")

    # Compute a quick debt-to-income ratio
    dti = (result.existing_emis / result.monthly_income) * 100 if result.monthly_income > 0 else 0
    print(f"   📊 Debt-to-Income Ratio: {dti:.1f}%")
    print("-" * 60)


👤 Parsed Applicant: Amit Verma
   Type: <class '__main__.LoanApplicant'> (Pydantic model — fully validated!)
   Age: 34 | Income: ₹185,000
   Employer: Infosys | Role: Senior Software Engineer
   Loan: ₹7,500,000 (home)
   Existing EMIs: ₹15,000 | CIBIL: 780
   📊 Debt-to-Income Ratio: 8.1%
------------------------------------------------------------

👤 Parsed Applicant: Sunita Reddy
   Type: <class '__main__.LoanApplicant'> (Pydantic model — fully validated!)
   Age: 28 | Income: ₹120,000
   Employer: Flipkart | Role: Product Manager
   Loan: ₹500,000 (personal)
   Existing EMIs: ₹0 | CIBIL: None
   📊 Debt-to-Income Ratio: 0.0%
------------------------------------------------------------


In [ ]:
result

LoanApplicant(full_name='Sunita Reddy', age=28, monthly_income=120000.0, existing_emis=0.0, loan_amount_requested=500000.0, loan_purpose='personal', employment=EmploymentDetails(employer='Flipkart', designation='Product Manager', years_in_current_job=2.5), cibil_score=None)

> **Key Difference vs JsonOutputParser:** `PydanticOutputParser` returns a **Pydantic model instance** (not a dict). This means you get attribute access (`result.full_name`), type validation (age is int, not str), default values, nested models, and custom validators. Use it when data quality matters.

---
## 10. EnumOutputParser — Classification into Fixed Categories

### Business Scenario: Customer Query Department Router
A bank's support system classifies incoming queries into exactly one of a fixed set of departments. The output **must** be one of the allowed values — no creative variations, no extra text. `EnumOutputParser` constrains the LLM to a predefined enum.

In [ ]:
from langchain_core.output_parsers import EnumOutputParser
from enum import Enum

class Department(str, Enum):
    LOANS = "loans"
    CREDIT_CARDS = "credit_cards"
    SAVINGS_ACCOUNTS = "savings_accounts"
    FRAUD = "fraud"
    GENERAL = "general"

enum_parser = EnumOutputParser(enum=Department)

print("📋 Format instructions:")
print(enum_parser.get_format_instructions())

ModuleNotFoundError: No module named 'langchain_core.output_parsers.enum'

In [ ]:
route_prompt = PromptTemplate(
    template="Classify this customer query into the correct department.\n\n"
             "Query: {query}\n\n{format_instructions}",
    input_variables=["query"],
    partial_variables={"format_instructions": enum_parser.get_format_instructions()}
)

route_chain = route_prompt | llm | enum_parser

test_queries = [
    "I want to know the interest rate on a home loan",
    "Someone made a ₹50,000 transaction on my card that I didn't authorize",
    "How do I increase my credit card limit?",
    "I want to open a fixed deposit",
    "What are your branch working hours on Saturday?"
]

for q in test_queries:
    dept = route_chain.invoke({"query": q})
    print(f"  {dept.value:20s} ← {q}")

> **When to use:** Classification tasks where the output MUST be from a fixed set — department routing, sentiment (positive/negative/neutral), priority levels (P1/P2/P3), approval status (approved/rejected/pending).

---
## 11. OutputFixingParser — Auto-Fix Malformed LLM Output

### Business Scenario: Resilient Data Pipeline
LLMs sometimes produce output that almost-but-not-quite matches the expected format — extra text before JSON, missing quotes, trailing commas. `OutputFixingParser` wraps another parser and **uses a second LLM call** to fix the output if the first parse fails.

In [ ]:
from langchain_core.output_parsers import OutputFixingParser

# Our target schema
class ProductReview(BaseModel):
    sentiment: str = Field(description="positive, negative, or neutral")
    rating: int = Field(description="Rating from 1 to 5")
    summary: str = Field(description="One-sentence summary")

base_parser = PydanticOutputParser(pydantic_object=ProductReview)

# Wrap it with OutputFixingParser
fixing_parser = OutputFixingParser.from_llm(parser=base_parser, llm=llm)

# --- Simulate a malformed LLM response ---
# This is what might come back from a less capable model or a prompt that didn't work perfectly
malformed_output = """Here is my analysis:
{
  sentiment: "positive",
  rating: 4,
  summary: "Great product with fast delivery"
}
Hope this helps!"""

print("❌ Attempting to parse malformed output with BASE parser:")
try:
    result = base_parser.parse(malformed_output)
    print(f"   Parsed: {result}")
except Exception as e:
    print(f"   Failed: {type(e).__name__}: {str(e)[:100]}...")

print("\n✅ Attempting with FIXING parser (auto-corrects via LLM):")
try:
    result = fixing_parser.parse(malformed_output)
    print(f"   Parsed: {result}")
    print(f"   Type: {type(result)}")
    print(f"   Sentiment: {result.sentiment} | Rating: {result.rating}")
except Exception as e:
    print(f"   Still failed: {e}")

ImportError: cannot import name 'OutputFixingParser' from 'langchain_core.output_parsers' (/usr/local/lib/python3.12/dist-packages/langchain_core/output_parsers/__init__.py)

> **How it works:** On parse failure, `OutputFixingParser` sends the malformed output + the error message + the original format instructions to the LLM, asking it to fix the output. It adds latency (extra LLM call) but dramatically improves reliability. Use it as a safety net in production.

---
## 12. `.with_structured_output()` — Native Model-Level Structured Output

### Business Scenario: Insurance Claim Classification API
For models that natively support structured output (OpenAI, Anthropic, Gemini), `.with_structured_output()` is the **recommended approach** — it's more reliable than prompt-based parsing because the model itself enforces the schema.

> This is not an "output parser" in the traditional sense — it uses the model's function calling / tool-use capability to guarantee schema compliance.

In [ ]:
# Define the schema as a Pydantic model
class ClaimClassification(BaseModel):
    claim_type: str = Field(description="One of: auto, health, property, life, travel")
    severity: str = Field(description="One of: low, medium, high, critical")
    estimated_amount: float = Field(description="Estimated claim amount in INR")
    requires_investigation: bool = Field(description="Whether the claim needs fraud investigation")
    summary: str = Field(description="One-sentence claim summary")

# .with_structured_output() — the model enforces the schema at generation time
structured_llm = llm.with_structured_output(ClaimClassification)

prompt = ChatPromptTemplate.from_template(
    "Classify this insurance claim:\n\nClaim: {claim_text}"
)

# Note: no parser needed in the chain — the model returns a Pydantic object directly
structured_chain = prompt | structured_llm

claims = [
    "Rear-ended at a signal. Bumper cracked, tail light broken. Other driver admitted fault. Repair estimate ₹45,000.",
    "Emergency appendectomy — 3 night hospital stay, surgery, medications. Total bill ₹2.8 lakhs. All receipts attached.",
    "Claiming ₹12 lakhs for water damage to my ground floor apartment. Happened during last month's floods. No photos available."
]

for claim in claims:
    result = structured_chain.invoke({"claim_text": claim})
    print(f"\n📋 Claim: {claim[:60]}...")
    print(f"   Type: {type(result)}")
    print(f"   Category: {result.claim_type} | Severity: {result.severity}")
    print(f"   Amount: ₹{result.estimated_amount:,.0f} | Investigation: {result.requires_investigation}")
    print(f"   Summary: {result.summary}")
    print("-" * 60)

/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ClaimClassification(claim...driver admitted fault.'), input_type=ClaimClassification])
  return self.__pydantic_serializer__.to_python(



📋 Claim: Rear-ended at a signal. Bumper cracked, tail light broken. O...
   Type: <class '__main__.ClaimClassification'>
   Category: auto | Severity: medium
   Amount: ₹45,000 | Investigation: False
   Summary: Rear-ended at signal causing bumper and tail light damage with repair estimate of ₹45,000; other driver admitted fault.
------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ClaimClassification(claim...totaling ₹2.8 lakhs.'), input_type=ClaimClassification])
  return self.__pydantic_serializer__.to_python(



📋 Claim: Emergency appendectomy — 3 night hospital stay, surgery, med...
   Type: <class '__main__.ClaimClassification'>
   Category: health | Severity: high
   Amount: ₹280,000 | Investigation: False
   Summary: Emergency appendectomy with 3-night hospital stay, surgery, and medications totaling ₹2.8 lakhs.
------------------------------------------------------------

📋 Claim: Claiming ₹12 lakhs for water damage to my ground floor apart...
   Type: <class '__main__.ClaimClassification'>
   Category: property | Severity: high
   Amount: ₹1,200,000 | Investigation: True
   Summary: Claim for ₹12 lakhs due to water damage to ground floor apartment during floods, no photos provided.
------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ClaimClassification(claim...s, no photos provided.'), input_type=ClaimClassification])
  return self.__pydantic_serializer__.to_python(


> **When to use `.with_structured_output()` vs Output Parsers:**
> - Use `.with_structured_output()` when your model supports it (OpenAI, Anthropic, Gemini) — it's more reliable because the model enforces the schema at generation time.
> - Use `PydanticOutputParser` / `JsonOutputParser` when working with models that don't support native structured output, or when you need the `get_format_instructions()` pattern for prompt injection.

---
## 11. XML Tagging via System Prompt — No Parser Library Needed

Some providers (Claude in particular) follow XML-tag instructions extraordinarily reliably via system prompt alone — no parser library, no schema registration, no `format_instructions` injection. This is the cheapest production pattern when:
- The output mixes prose and structured data (e.g., a long answer with `<citation>` tags)
- You don't want to drag in a parser dependency
- You want regex-parseable output that any language can consume

When the schema is rigid and every-field-every-time, prefer Pydantic + `.with_structured_output()` (Section 10).


In [ ]:
# --- Claude with XML-tagged system prompt ---
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage

xml_system = '''You analyze retail-bank loan applications and return your assessment in XML tags.

Respond using EXACTLY this format:
<assessment>
  <risk_level>low|medium|high</risk_level>
  <key_factors>
    <factor>factor 1</factor>
    <factor>factor 2</factor>
  </key_factors>
  <recommendation>your recommendation</recommendation>
</assessment>

Do not include any text outside the <assessment> tags.'''

claude = ChatAnthropic(model='claude-haiku-4-5', temperature=0)

response = claude.invoke([
    SystemMessage(content=xml_system),
    HumanMessage(content=(
        'Applicant: 32yo, ₹1.2L/mo income, CIBIL 762, '
        '₹15K existing EMIs, applying for ₹50L home loan over 20 years.'
    )),
])

print(response.content)


In [ ]:
# --- Parse with regex (no library — XML tagging is dependency-free) ---
import re

xml = response.content
risk = re.search(r'<risk_level>(.*?)</risk_level>', xml).group(1).strip()
factors = [f.strip() for f in re.findall(r'<factor>(.*?)</factor>', xml, re.DOTALL)]
recommendation = re.search(r'<recommendation>(.*?)</recommendation>', xml, re.DOTALL).group(1).strip()

print(f'Risk level     : {risk}')
print(f'Key factors    : {factors}')
print(f'Recommendation : {recommendation}')

# Production tip: lxml.etree or xml.etree.ElementTree for nested or repeated structures.


---
## 13. Multi-Step Output Chaining — Parsed Output as Input to Next Step

### Business Scenario: E-Commerce Returns Processing Pipeline
A returns pipeline that: (1) extracts return request details as structured data, (2) applies business rules to decide the resolution, (3) generates a customer email. Each step's **parsed output feeds into the next step** using LCEL.

This demonstrates the real power of output parsers — turning LLM text into machine-readable data that drives downstream logic.

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ═══ Step 1: Extract return request details ═══
class ReturnRequest(BaseModel):
    customer_name: str = Field(description="Customer's name")
    order_id: str = Field(description="Order ID")
    product: str = Field(description="Product name")
    reason: str = Field(description="Return reason: defective, wrong_item, not_as_described, changed_mind, other")
    days_since_delivery: int = Field(description="Days since the product was delivered")
    purchase_amount: float = Field(description="Original purchase amount in INR")

extract_parser = PydanticOutputParser(pydantic_object=ReturnRequest)

extract_prompt = PromptTemplate(
    template="Extract return request details from this customer message.\n\n"
             "Message: {message}\n\n{format_instructions}",
    input_variables=["message"],
    partial_variables={"format_instructions": extract_parser.get_format_instructions()}
)

extract_chain = extract_prompt | llm | extract_parser

print("Step 1 (extraction) ready ✅")

In [ ]:
# ═══ Step 2: Apply business rules (pure Python — no LLM needed) ═══
def apply_return_policy(request: ReturnRequest) -> dict:
    """Apply business rules to determine resolution. Returns enriched dict."""
    data = request.model_dump()  # Convert Pydantic model to dict

    # Business rules
    if request.days_since_delivery > 30:
        data["resolution"] = "REJECTED"
        data["resolution_reason"] = "Return window (30 days) has expired."
        data["refund_amount"] = 0
    elif request.reason in ["defective", "wrong_item"]:
        data["resolution"] = "FULL_REFUND"
        data["resolution_reason"] = f"Eligible for full refund due to {request.reason.replace('_', ' ')}."
        data["refund_amount"] = request.purchase_amount
    elif request.reason == "changed_mind":
        data["resolution"] = "PARTIAL_REFUND"
        data["resolution_reason"] = "Changed mind returns incur 15% restocking fee."
        data["refund_amount"] = round(request.purchase_amount * 0.85, 2)
    else:
        data["resolution"] = "FULL_REFUND"
        data["resolution_reason"] = "Eligible for full refund."
        data["refund_amount"] = request.purchase_amount

    return data

print("Step 2 (business rules) ready ✅")

In [ ]:
# ═══ Step 3: Generate customer email using enriched data ═══
email_prompt = ChatPromptTemplate.from_template(
    """Draft a professional, empathetic customer email for this return resolution.

Customer: {customer_name}
Order: {order_id}
Product: {product}
Resolution: {resolution}
Reason: {resolution_reason}
Refund Amount: ₹{refund_amount}

Keep the email under 100 words. Sign off as "Customer Experience Team"."""
)

email_chain = email_prompt | llm | StrOutputParser()

print("Step 3 (email generation) ready ✅")

In [ ]:
# ═══ Full Pipeline: Extract → Business Rules → Email ═══
full_returns_pipeline = (
    extract_chain                              # Step 1: LLM → PydanticOutputParser → ReturnRequest
    | RunnableLambda(apply_return_policy)       # Step 2: Python business rules → enriched dict
    | email_chain                               # Step 3: dict → prompt → LLM → email text
)

# Test with different return scenarios
return_messages = [
    "Hi, I'm Kavita Joshi. My order ORD-2024-55321 for the Sony WH-1000XM5 headphones (₹24,990) "
    "arrived 5 days ago but the left earcup is making a buzzing noise. Clearly defective. Please process a return.",

    "Hello, this is Rohan Agarwal. I bought a Levi's denim jacket (₹4,500) two weeks ago (order ORD-2024-88910). "
    "Honestly I just changed my mind — it doesn't suit my wardrobe. Can I return it?",

    "My name is Deepa Nair. I received order ORD-2024-11200 for a KitchenAid mixer (₹32,000) over 6 weeks ago. "
    "The motor stopped working yesterday. I want a refund."
]

for msg in return_messages:
    print(f"\n{'=' * 70}")
    print(f"📩 Customer message: {msg[:80]}...")

    # Run step 1 separately to show intermediate output
    extracted = extract_chain.invoke({"message": msg})
    print(f"\n📋 Step 1 — Extracted: {extracted.customer_name} | {extracted.reason} | {extracted.days_since_delivery} days")

    # Run step 2 separately
    enriched = apply_return_policy(extracted)
    print(f"⚙️  Step 2 — Resolution: {enriched['resolution']} | Refund: ₹{enriched['refund_amount']:,.0f}")

    # Run full pipeline
    email = full_returns_pipeline.invoke({"message": msg})
    print(f"\n📧 Step 3 — Email:\n{email}")

> **Key Insight:** This pipeline demonstrates the core pattern: **LLM + Parser → structured data → Python logic → LLM + Parser → output**. The output parsers transform unstructured text into actionable data that Python business rules can operate on. This is how production GenAI systems work — LLMs handle the fuzzy text understanding, parsers provide the structured bridge, and Python handles the deterministic business logic.

---
## 14. Comparison: Using Groq (Llama 3.3) with Output Parsers

### Does open-source follow format instructions as well as GPT-4.1?
Let's run the same structured extraction task on both providers and compare.

In [ ]:
import time

# Use the same triage schema from Section 8
triage_prompt_compare = PromptTemplate(
    template="You are a medical triage assistant. Analyze the symptoms and respond with structured JSON.\n\n"
             "Symptoms: {symptoms}\n\n{format_instructions}",
    input_variables=["symptoms"],
    partial_variables={"format_instructions": json_parser_schema.get_format_instructions()}
)

# Chains with different providers
chain_openai = triage_prompt_compare | llm | json_parser_schema
chain_groq = triage_prompt_compare | llm_groq | json_parser_schema

test_input = {"symptoms": "Persistent cough for 3 weeks, mild fever in the evenings, weight loss, night sweats"}

for name, chain in [("OpenAI gpt-4.1-mini", chain_openai), ("Groq Llama-3.3-70B", chain_groq)]:
    start = time.time()
    try:
        result = chain.invoke(test_input)
        elapsed = round(time.time() - start, 2)
        print(f"\n🤖 {name} ({elapsed}s):")
        print(f"   Severity: {result.get('severity')} | Urgency: {result.get('urgency_minutes')}min")
        print(f"   Conditions: {result.get('likely_conditions')}")
        print(f"   Action: {result.get('recommended_action')}")
        print(f"   ✅ JSON parsed successfully")
    except Exception as e:
        elapsed = round(time.time() - start, 2)
        print(f"\n🤖 {name} ({elapsed}s): ❌ Parse failed — {e}")
    print("-" * 60)

> **Production Tip:** Open-source models sometimes struggle with strict format instructions (adding extra text, wrong JSON keys). The `OutputFixingParser` from §11 is especially valuable when using open-source models in production.

---
## 15. Quick Reference: Output Parser Selection Guide

| Need | Parser | Returns | Format Instructions? |
|------|--------|---------|---------------------|
| Plain text | `StrOutputParser` | `str` | No |
| Comma-separated list | `CommaSeparatedListOutputParser` | `list[str]` | Yes |
| Date/time | `DatetimeOutputParser` | `datetime` | Yes |
| Flat dictionary | `StructuredOutputParser` | `dict` | Yes |
| Flexible JSON | `JsonOutputParser` | `dict` | Yes |
| Validated typed object | `PydanticOutputParser` | Pydantic model | Yes |
| Fixed category | `EnumOutputParser` | Enum value | Yes |
| Auto-fix failures | `OutputFixingParser` | (wraps another) | No (uses inner) |
| Native model schema | `.with_structured_output()` | Pydantic / dict | N/A (model-level) |

### Decision Tree
```
Do you need structured output?
├── No → StrOutputParser
└── Yes
    ├── Just a list? → CommaSeparatedListOutputParser
    ├── Just a date? → DatetimeOutputParser
    ├── Fixed category? → EnumOutputParser
    └── Complex structure?
        ├── Model supports structured output? → .with_structured_output()
        ├── Need type validation? → PydanticOutputParser
        ├── Need flexible JSON? → JsonOutputParser
        └── Quick flat dict? → StructuredOutputParser
```

---
## 16. Conclusion & Next Steps

### What We Covered

| Section | Parser | Business Use Case |
|---------|--------|------------------|
| §4 | `StrOutputParser` | Bank email reply + streaming |
| §5 | `CommaSeparatedListOutputParser` | E-commerce product tag generation |
| §6 | `DatetimeOutputParser` | Insurance policy date extraction |
| §7 | `StructuredOutputParser` | Sales call CRM data extraction |
| §8 | `JsonOutputParser` | Healthcare triage API (schema vs free-form) |
| §9 | `PydanticOutputParser` | Loan application pre-screening (nested models) |
| §10 | `EnumOutputParser` | Bank query department classification |
| §11 | `OutputFixingParser` | Auto-fix malformed LLM output |
| §12 | `.with_structured_output()` | Insurance claim classification (native) |
| §13 | **Multi-step chaining** | E-commerce returns pipeline (extract → rules → email) |
| §14 | **Provider comparison** | OpenAI vs Groq format compliance |

### Production Insights
- **Prefer `.with_structured_output()`** for models that support it (OpenAI, Anthropic, Gemini) — it's more reliable than prompt-based parsing
- **Always wrap with `OutputFixingParser`** when using open-source models or when parse reliability is critical
- **`PydanticOutputParser`** is the gold standard for production — nested models, type validation, default values
- **Output parsers + LCEL** enable the core production pattern: LLM → structured data → Python business logic → LLM → output

### Try on Your Own
1. Create a **multi-parser pipeline** that extracts a list (`CommaSeparatedListOutputParser`), then feeds each item into a `PydanticOutputParser` chain for detailed analysis
2. Build a **resume parser** using `PydanticOutputParser` with nested models (personal info, education, experience, skills)
3. Test `OutputFixingParser` with Groq/Llama — does it improve parse success rates vs bare `PydanticOutputParser`?
4. Create an **invoice data extractor** using `StructuredOutputParser` and chain it into a tax calculation step
5. Use `.with_structured_output()` to build a **real-time stock screener** that returns buy/hold/sell with validated confidence scores

---

**Next Lab:** Module 7 — Prompt Engineering Applications with LangChain 🔧

---
## 13. Conclusion & Key Takeaways

### What We Covered

| Concept | Takeaway |
|---|---|
| **StrOutputParser** | Clean string output + streaming. Foundation of every chain. |
| **CommaSeparatedList / Datetime / Structured** | Single-purpose parsers with built-in `format_instructions`. |
| **JsonOutputParser vs PydanticOutputParser** | Both inject schema; Pydantic returns typed objects, JSON returns plain dicts. |
| **EnumOutputParser** | Classification with a fixed label set — model can't invent labels. |
| **OutputFixingParser** | Auto-retry on parse failure — useful for legacy models, code smell on modern ones. |
| **Pydantic + .with_structured_output()** | Modern preferred path on tool-calling-capable models. Provider-validated. |
| **XML tagging via system prompt** | Library-free; great for Claude on prose-mixed outputs. |
| **Pipeline pattern** | LLM → parser → business rules → LLM with typed objects throughout. Production-ready. |

**Next Lab:** Lab 2.5 — Prompt Caching Across Providers (OpenAI · Anthropic · Gemini) 💸


## 14. Stretch Exercise (Optional)

1. Convert the loan-application XML pattern to a Pydantic model with `.with_structured_output()`. Compare reliability over 20 calls.
2. Build an `OutputFixingParser` chain that gracefully falls back to regex parsing if the fixer also fails — three-tier defense.
3. Add a nested-schema example: `Customer` with a list of `Account` objects, each with a list of `Transaction` objects. Test how each parser handles depth.
4. Use `EnumOutputParser` to build a 12-class banking-intent router; benchmark against a JSON-mode classifier on accuracy and latency.
5. Build the same triage pipeline (Section 12) using XML tagging instead of JSON; compare token counts and parse-failure rates.


---

## Interview Preparation

The questions below mirror what client interviewers commonly ask about the topics in this lab. Use the hint to think through the answer first; use the sketch only to verify your reasoning.

---

**Q1. PydanticOutputParser vs JsonOutputParser vs `.with_structured_output()` — when to use each?**

*Hint:* Three layers: prompt-only, prompt + parse, native tool-calling.

*Answer sketch:* PydanticOutputParser injects `format_instructions` into the prompt and parses to a typed model — works on any LLM. JsonOutputParser does the same but returns a dict, no typing. `.with_structured_output()` uses the provider's native tool-calling / structured-output API — provider-validated, highest reliability, but requires a model that supports it. Default: `.with_structured_output()` when supported; PydanticOutputParser otherwise.

---

**Q2. When does OutputFixingParser save you, and when is it a code smell?**

*Hint:* Recovery vs root-cause.

*Answer sketch:* Saves you with legacy / open-source models that don't reliably emit valid JSON, or as a graceful one-time retry on transient flakiness. Code smell when used to paper over a poorly-written prompt — fix the prompt or switch to a model with native structured output instead. Each fix call is an extra LLM round trip.

---

**Q3. Why use XML tagging when JSON exists — what's the trade-off?**

*Hint:* Output shape and parser dependency.

*Answer sketch:* XML tagging shines for prose-mixed structured output (long answer with `<citation>` tags inline) and is parseable with regex — no library dependency. JSON shines for clean records (every field, every time) and benefits from native structured-output APIs. Claude in particular follows XML tags extremely reliably via system prompt alone.

---

**Q4. How does EnumOutputParser differ from a Pydantic Literal field?**

*Hint:* Both restrict to a fixed set — but at different layers.

*Answer sketch:* EnumOutputParser injects "valid options are X, Y, Z" into the prompt and validates the raw output. A Pydantic `Literal['X', 'Y', 'Z']` field used with `.with_structured_output()` does the same constraint but at the provider's tool-calling layer — usually more reliable. Functionally equivalent for the trainee; mechanically different (prompt-side vs API-side enforcement).

---

**Q5. What happens at the prompt level when you use `format_instructions`?**

*Hint:* Look at the parser's `.get_format_instructions()`.

*Answer sketch:* The parser generates a string describing the expected output format (often JSON schema or a list spec) and you inject it into the prompt as a partial variable. The model sees these instructions verbatim and shapes its output accordingly. The same parser then validates and converts the model's output. So the parser is responsible for *both ends* — prompt side and parse side.

---

**Q6. When would you skip the parser entirely and parse with regex?**

*Hint:* Output is XML-tagged or follows a tight pattern.

*Answer sketch:* When the output is XML-tagged via system prompt, when the output follows a fixed template (e.g., `RESULT: ...`), or when you want zero parser-library dependency. Regex stays simple as long as the structure stays simple — break out a real parser when you need nested or repeated fields with attributes.

---

**Q7. How would you handle a nested schema (object with arrays of objects)?**

*Hint:* Pydantic handles arbitrary nesting.

*Answer sketch:* Define each layer as its own Pydantic model and reference it in the parent: `class Order(BaseModel): items: list[OrderItem]`. Then `.with_structured_output(Order)` (when supported) or `PydanticOutputParser(pydantic_object=Order)` will handle nesting automatically. The provider serializes the full schema, validates the output, and returns a fully-typed nested model.

